# Challenge W4: NLP - Fake News
## Steps:
    1. Import data
    2. EDA
    3. Data Splitting
    4. Text Preprocessing 
    5. Feature Extraction
    6. Model Training

##  Libraries:

In [132]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
#import seaborn as sns

import re
import string

from sklearn.model_selection import train_test_split

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# 1. Load data

In [133]:
df = pd.read_csv("./dataset/data.csv")

In [134]:
df.shape
df.head()       
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 39942 entries, 0 to 39941
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   label    39942 non-null  int64 
 1   title    39942 non-null  object
 2   text     39942 non-null  object
 3   subject  39942 non-null  object
 4   date     39942 non-null  object
dtypes: int64(1), object(4)
memory usage: 1.5+ MB


# 2. EDA

In [135]:
df.isnull().sum() # missing values?

label      0
title      0
text       0
subject    0
date       0
dtype: int64

In [136]:
print("Duplicate rows:", df.duplicated().sum()) # Duplicates?


Duplicate rows: 201


In [137]:
df.sample(5, random_state=42)

label  \
6524       1   
30902      0   
36459      0   
9801       1   
25638      0   

                                                                                               title  \
6524                                Oil business seen in strong position as Trump tackles tax reform   
30902        WHOA! COLLEGE SNOWFLAKE FREAKS OUT: Screams For Two Minutes Over A Trump Sign On Campus   
36459  CRONY CORRUPT POLITICS: Obama Admin BLOCKED FBI From Doing A Clinton Foundation Investigation   
9801                                   Cruz campaign vetting Fiorina as a possible VP pick: ABC News   
25638         Minnesota Woman Writes Amazing F*ck Off Letter To Men Who Want To Ban Abortion (IMAGE)   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     

In [138]:
print("Unique subjects:", df['subject'].nunique())
df['subject'].value_counts()

Unique subjects: 6


subject
politicsNews       11272
News                9050
worldnews           8727
politics            6841
left-news           2482
Government News     1570
Name: count, dtype: int64

In [139]:
pd.crosstab(df['subject'], df['label']) # Because subject and label are correlated we should drop "subject" column to avoid data leakage.

label,0,1
subject,,
Government News,1570,0
News,9050,0
left-news,2482,0
politics,6841,0
politicsNews,0,11272
worldnews,0,8727


# 3. TRAIN TEST VALIDATION SPLIT

In [140]:
# Dropping columns subject and date as the have been shown to not carry important information
df = df.drop(columns=['subject','date']) 

In [141]:
df = df.drop_duplicates() # Dropping 201 duplicates that appeared once subject and date were removed (meaning that they had same title and/or text and we 
                            #could fall into data leakage)
print(df.shape) 

(36429, 3)


In [142]:

# Split into features and target
# Adjust the target column name to match your dataset (e.g. 'label', 'class', 'Spam/Ham')
X = df['title'] + ' ' + df['text']   # <-- replace 'label' with your actual target column
y = df['label']

# First split: 70% train, 30% temporary
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

# Second split: Split the remaining 30% into 15% validation and 15% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

# Display dataset shapes
print(f"Train shape:      {X_train.shape}")
print(f"Validation shape: {X_val.shape}")
print(f"Test shape:       {X_test.shape}")

# Display class balance
print("\nTrain class balance:")
print(y_train.value_counts(normalize=True))

print("\nValidation class balance:")
print(y_val.value_counts(normalize=True))

print("\nTest class balance:")
print(y_test.value_counts(normalize=True))

Train shape:      (25500,)
Validation shape: (5464,)
Test shape:       (5465,)

Train class balance:
label
1    0.543216
0    0.456784
Name: proportion, dtype: float64

Validation class balance:
label
1    0.543192
0    0.456808
Name: proportion, dtype: float64

Test class balance:
label
1    0.543275
0    0.456725
Name: proportion, dtype: float64


# 4. Text Preprocessing 
    - we will combine it into a reusable function 

In [143]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = re.sub(r'^[A-Z][A-Za-z\.,\'\s]*\(Reuters\)\s*-\s*', '', text)
    text = re.sub(r'\breuters\b', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\[.*?\]', '', text)
    text = re.sub(r'\b(via|h/t)\s*:\s*\S.*$', '', text, flags=re.IGNORECASE)
    text = re.sub(r'read more\s*:\s*\S.*$', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\bimage via\b.*$', '', text, flags=re.IGNORECASE)
    text = text.lower() # Lowercasing
    text = re.sub(r'[^a-z\s]', '', text) # Remove special characters
    tokens = word_tokenize(text) # Tokenization
    tokens = [word for word in tokens if word not in stop_words] # stopword removal
    tokens = [lemmatizer.lemmatize(word) for word in tokens] # Lemmatization
    return ' '.join(tokens)


Applying the text processing to X train, test and val

In [144]:
X_train_clean = X_train.apply(clean_text)
X_test_clean = X_test.apply(clean_text)
X_val_clean = X_val.apply(clean_text)

# 5. Feature extraction - Baseline model
 - TF-IDF
 - n-grams (with TF-IDF)
 

In [145]:
# TF-IDF

from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(max_features=5000)

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train_clean)
X_val_tfidf = tfidf_vectorizer.transform(X_val_clean)
X_test_tfidf = tfidf_vectorizer.transform(X_test_clean)

In [146]:
# n-grams with TF-IDF

tfidf_ngram_vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))

X_train_tfidf_ngram = tfidf_ngram_vectorizer.fit_transform(X_train_clean)
X_val_tfidf_ngram = tfidf_ngram_vectorizer.transform(X_val_clean)
X_test_tfidf_ngram = tfidf_ngram_vectorizer.transform(X_test_clean)

print(X_train_tfidf_ngram.shape)

(25500, 5000)


# 6. Model training - Baseline model
    - Logistic regression

In [147]:


from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Logistic regression with TDF-IDF

lr_tfidf = LogisticRegression(max_iter=1000, random_state=42)
lr_tfidf.fit(X_train_tfidf, y_train)

y_val_pred_tfidf = lr_tfidf.predict(X_val_tfidf)

print("=== Logistic Regression + TF-IDF (unigrams) ===")
print("Accuracy:", accuracy_score(y_val, y_val_pred_tfidf))
print(classification_report(y_val, y_val_pred_tfidf))
print(confusion_matrix(y_val, y_val_pred_tfidf))

=== Logistic Regression + TF-IDF (unigrams) ===
Accuracy: 0.9789531478770132
              precision    recall  f1-score   support

           0       0.98      0.97      0.98      2496
           1       0.98      0.99      0.98      2968

    accuracy                           0.98      5464
   macro avg       0.98      0.98      0.98      5464
weighted avg       0.98      0.98      0.98      5464

[[2422   74]
 [  41 2927]]


In [148]:
# Logistic regression with n-grams (TD-IDF)

lr_ngram = LogisticRegression(max_iter=1000, random_state=42)
lr_ngram.fit(X_train_tfidf_ngram, y_train)

y_val_pred_ngram = lr_ngram.predict(X_val_tfidf_ngram)

print("\n=== Logistic Regression + TF-IDF (uni+bigrams) ===")
print("Accuracy:", accuracy_score(y_val, y_val_pred_ngram))
print(classification_report(y_val, y_val_pred_ngram))
print(confusion_matrix(y_val, y_val_pred_ngram))


=== Logistic Regression + TF-IDF (uni+bigrams) ===
Accuracy: 0.9807833089311859
              precision    recall  f1-score   support

           0       0.98      0.97      0.98      2496
           1       0.98      0.99      0.98      2968

    accuracy                           0.98      5464
   macro avg       0.98      0.98      0.98      5464
weighted avg       0.98      0.98      0.98      5464

[[2428   68]
 [  37 2931]]


In [149]:
feature_names = np.array(tfidf_ngram_vectorizer.get_feature_names_out())
coefs = lr_ngram.coef_[0]

top_positive = np.argsort(coefs)[-20:][::-1]   # pushes prediction toward label 1 (real)
top_negative = np.argsort(coefs)[:20]           # pushes prediction toward label 0 (fake)

print("Top words/n-grams predicting REAL (1):")
for i in top_positive:
    print(f"{feature_names[i]:20s} {coefs[i]:.3f}")

print("\nTop words/n-grams predicting FAKE (0):")
for i in top_negative:
    print(f"{feature_names[i]:20s} {coefs[i]:.3f}")

Top words/n-grams predicting REAL (1):
said                 17.508
president donald     7.026
tuesday              5.832
wednesday            5.757
thursday             4.814
friday               4.696
monday               4.365
washington           3.894
presidential         3.754
nov                  3.647
representative       3.403
told                 3.401
president barack     3.313
statement            3.304
told reporter        3.232
minister             3.227
said statement       3.219
dont                 3.209
im                   3.092
spokesman            3.025

Top words/n-grams predicting FAKE (0):
gop                  -6.407
hillary              -5.101
obama                -5.010
president trump      -4.873
even                 -4.582
video                -4.184
like                 -4.179
breaking             -3.720
mr                   -3.720
president obama      -3.683
rep                  -3.662
american             -3.640
sen                  -3.626
america         

In [150]:
pd.set_option('display.max_colwidth', None)

errors = X_val[y_val != y_val_pred_ngram]
print(errors.head(10))

33504                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   